In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.tree import export_graphviz
import pydotplus
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

import os
import re
import json 

In [ ]:
def get_error_estimation_rf(data, days_to_remove, days_to_estimate, link, normalize_data, save_figures=False, save_path=None, show_figs = False):
    
    print(f'Estimating for link {link}')
        
    if normalize_data:
        max_value = max(data['Error'].values)
        data['Error'] = np.floor((data['Error'].values /max_value) * 100)
    # else:
    #     data['Error'] = data['Error'].values 

    data_df = data
    N = len(data['Error'])
    # Convert the date column to datetime
    data_df['Date'] = pd.to_datetime(data_df['Date'])

    # Extract the numerical representation of the date and additional features
    data_df['date_ordinal'] = data_df['Date'].map(datetime.toordinal)
    data_df['day_of_week'] = data_df['Date'].dt.dayofweek
    data_df['month'] = data['Date'].dt.month
    data_df['day_of_month'] = data_df['Date'].dt.day

    # Sort the dataframe by date
    data_df = data_df.sort_values(by='Date')

        # Split the data
    if days_to_remove == 0:
        past_data_values = data['Error']
        past_dates = data['Date']
        last_date = past_dates.iloc[-1]
        future_data_values =  pd.date_range(start=last_date, periods=N_estimate + 1, freq='D')[1:]
    else: 
        # Separate the last 5 days from the dataset
        # past_dates = data_df.tail(days_to_remove)
        past_df = data_df.iloc[:-days_to_remove]
        future_df = data_df.iloc[-days_to_remove:]
        last_date = past_df['Date'].iloc[-1]

    # Prepare the features (X) and target (y)
    X = past_df[['date_ordinal', 'day_of_week', 'month', 'day_of_month']]
    y = past_df['Error']

    # Train the Random Forest model
        # n_estimators: Number of trees
        # criterion: Measure the quality of a split
    model = RandomForestRegressor(n_estimators=100, criterion='squared_error', random_state=42)
    model.fit(X, y)

    # Prepare the features for the last 5 days
    X_future_pred = future_df[['date_ordinal', 'day_of_week', 'month', 'day_of_month']]
    y_future_actual = future_df['Error']

    # Predict fidelity for the last 5 days
    y_past_pred = model.predict(X)
    y_future_pred = model.predict(X_future_pred)

    #Metrics
        # Past
    error_of_past_estimation = abs(y - y_past_pred)
    mean_error_past_estimation = np.mean(error_of_past_estimation)
    std_error_past_estimation = np.std(error_of_past_estimation)
    rmse_error_past_estimation = root_mean_squared_error(y, y_past_pred)
        
        # Future
    error_of_future_estimation = abs(y_future_actual - y_future_pred)
    mean_error_future_estimation = np.mean(error_of_future_estimation)
    std_error_future_estimation = np.std(error_of_future_estimation)
    rmse_error_future_estimation = root_mean_squared_error(y_future_actual, y_future_pred)
        
        # Complete
    
    complete_estimation = [ *y_past_pred, *y_future_pred]
    error_of_complete_estimation = abs(data['Error'] - complete_estimation)
    mean_error_complete_estimation = np.mean(error_of_complete_estimation)
    std_error_complete_estimation = np.std(error_of_complete_estimation)
    rmse_error_complete_estimation = root_mean_squared_error(data['Error'], complete_estimation)

    result = {
        link: {
            "Normalized_data": normalize_data,

            "Complete_dates": data['Date'],
            "Complete_data": data['Error'],
            "Complete_estimated_data": complete_estimation,

            "Complete_estimation_error": error_of_complete_estimation,
            "Complete_mean_estimation_error": mean_error_complete_estimation,
            "Complete_std_estimation_error": std_error_complete_estimation,
            "Complete_rmse_estimation_error": rmse_error_complete_estimation,
                       
            "Past_dates": past_df['Date'],
            "Last_date": last_date,
            "Past_data": past_df['Error'],
            "Past_estimated_data": y_past_pred,

            "Past_estimation_error": error_of_past_estimation,
            "Past_mean_estimation_error": mean_error_past_estimation,
            "Past_std_estimation_error": std_error_past_estimation,
            "Past_rmse_estimation_error": rmse_error_past_estimation,


            "Future_dates": future_df['Date'],
            "Future_data": future_df['Error'],
            "Future_estimated_data": y_future_pred,

            "Future_estimation_error": error_of_future_estimation,
            "Future_mean_estimation_error": mean_error_future_estimation,
            "Future_std_estimation_error": std_error_future_estimation,
            "Future_rmse_estimation_error": rmse_error_future_estimation
        }
    }

    if show_figs or save_figures:
        # Plotting
        plt.figure(figsize=(18, 30))

        #Complete data plot
        plt.subplot(9, 1, 1)
        plt.axvline(result[link]['Last_date'], color='r', linestyle=':', label='Forecast Start')
        plt.plot(result[link]['Complete_dates'], result[link]['Complete_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Complete_dates'], result[link]['Complete_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'Actual vs Estimated Data')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        #Complete error estimation plot
        plt.subplot(9, 1, 2)
        plt.axvline(result[link]['Last_date'], color='r', linestyle=':', label='Forecast Start')
        plt.plot(result[link]['Complete_dates'], result[link]['Complete_estimation_error'], label='Error of the Estimation', color='green')
        plt.axhline(result[link]['Complete_mean_estimation_error'], linestyle='dashed', label='Average Estimation Error', color='purple')
        plt.axhline(result[link]['Complete_rmse_estimation_error'], linestyle='dashed', label='Root Mean Squared Error', color='deeppink')
        plt.title(f'Prediction Error of Actual vs Estimated Data    Avg: {result[link]['Complete_mean_estimation_error']:.5f} RMSE:{result[link]['Complete_rmse_estimation_error']:.5f}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Complete upper and lower bounds
        plt.subplot(9, 1, 3)
        plt.axvline(result[link]['Last_date'], color='r', linestyle=':', label='Forecast Start')
        plt.plot(result[link]['Complete_dates'], result[link]['Complete_data']+result[link]['Complete_mean_estimation_error'], label='Upper Data Bound', color='teal')
        plt.plot(result[link]['Complete_dates'], result[link]['Complete_data']-result[link]['Complete_mean_estimation_error'], label='Lower Data Bound', color='orchid')
        plt.plot(result[link]['Complete_dates'], result[link]['Complete_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--')
        plt.title(f'Bounds of Actual vs Estimated Data  std: {result[link]['Complete_std_estimation_error']}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        #Future data plot
        plt.subplot(9, 1, 4)
        plt.plot(result[link]['Future_dates'], result[link]['Future_data'], label='Actual Data', color='royalblue',  marker='o')
        plt.plot(result[link]['Future_dates'], result[link]['Future_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--', marker='x')
        plt.title(f'Actual vs Estimated Data for the future {days_to_estimate} Days')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        #Future error estimation plot
        plt.subplot(9, 1, 5)
        plt.plot(result[link]['Future_dates'], result[link]['Future_estimation_error'], label='Error of the Estimation', color='green',  marker='o')
        plt.axhline(result[link]['Future_mean_estimation_error'], linestyle='dashed', label='Average Estimation Error', color='purple')
        plt.axhline(result[link]['Future_rmse_estimation_error'], linestyle='dashed', label='Root Mean Squared Error', color='deeppink')
        plt.title(f'Prediction Error of Actual vs Estimated Data for the future {days_to_estimate} Days     Avg: {result[link]['Future_mean_estimation_error']:.5f} RMSE:{result[link]['Future_rmse_estimation_error']:.5f}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Future upper and lower bounds
        plt.subplot(9, 1, 6)
        plt.plot(result[link]['Future_dates'], result[link]['Future_data']+result[link]['Future_mean_estimation_error'], label='Upper Data Bound', color='teal',  marker='o')
        plt.plot(result[link]['Future_dates'], result[link]['Future_data']-result[link]['Future_mean_estimation_error'], label='Lower Data Bound', color='orchid',  marker='o')
        plt.plot(result[link]['Future_dates'], result[link]['Future_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--',  marker='x')
        plt.title(f'Bounds of Actual vs Estimated Data for the future {days_to_estimate} Days   std: {result[link]['Future_std_estimation_error']}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()



        #Past data plot
        plt.subplot(9, 1, 7)
        plt.plot(result[link]['Past_dates'], result[link]['Past_data'], label='Actual Data', color='royalblue')
        plt.plot(result[link]['Past_dates'], result[link]['Past_estimated_data'], label='Estimated Data', color='darkorange', linestyle='--')
        plt.title(f'Actual vs Estimated Data for the past {N-days_to_estimate} Days')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        #Past error estimation plot
        plt.subplot(9, 1, 8)
        plt.plot(result[link]['Past_dates'], result[link]['Past_estimation_error'], label='Error of the Estimation', color='green')
        plt.axhline(result[link]['Past_mean_estimation_error'], linestyle='dashed', label='Average Estimation Error', color='purple')
        plt.axhline(result[link]['Past_rmse_estimation_error'], linestyle='dashed', label='Root Mean Squared Error', color='deeppink')
        plt.title(f'Prediction Error of Actual vs Estimated Data for the past {N-days_to_estimate} Days     Avg: {result[link]['Past_mean_estimation_error']:.5f}  RMSE:{result[link]['Past_rmse_estimation_error']:.5f}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()

        # Past upper and lower bounds
        plt.subplot(9, 1, 9)
        plt.plot(result[link]['Past_dates'], result[link]['Past_data']+result[link]['Past_mean_estimation_error'], label='Upper Data Bound', color='teal')
        plt.plot(result[link]['Past_dates'], result[link]['Past_data']-result[link]['Past_mean_estimation_error'], label='Lower Data Bound', color='orchid')
        plt.plot(result[link]['Past_dates'], result[link]['Past_estimated_data'], label='Estimated Data', color='chocolate', linestyle='--')
        plt.title(f'Bounds of Actual vs Estimated Data for the past {N-days_to_estimate} Days   std: {result[link]['Past_std_estimation_error']}')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.legend()
        plt.grid()


        plt.tight_layout()
        if save_figures:
            plt.savefig(save_path)
            plt.close()
        
        else: 
            plt.show()


    result[link]["Complete_dates"] = result[link]["Complete_dates"].to_list()
    result[link]["Past_dates"] = result[link]["Past_dates"].to_list()
    result[link]["Future_dates"] = result[link]["Future_dates"].to_list()
    
    for key in result:
        result[key]['Complete_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Complete_dates']]
        result[key]['Past_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Past_dates']]
        result[key]['Future_dates'] = [date.strftime('%Y-%m-%d') for date in result[key]['Future_dates']]
        result[key]['Last_date'] = result[key]['Last_date'].strftime('%Y-%m-%d')

    result[link]["Complete_data"] = result[link]["Complete_data"].tolist()
    # result[link]["Complete_estimated_data"] = result[link]["Complete_estimated_data"].tolist()
    result[link]["Complete_estimation_error"] = result[link]["Complete_estimation_error"].tolist()
    result[link]["Past_data"] = result[link]["Past_data"].tolist()
    result[link]["Past_estimated_data"] = result[link]["Past_estimated_data"].tolist()
    result[link]["Past_estimation_error"] = result[link]["Past_estimation_error"].tolist()
    result[link]["Future_data"] = result[link]["Future_data"].tolist()
    result[link]["Future_estimated_data"] = result[link]["Future_estimated_data"].tolist()
    result[link]["Future_estimation_error"] = result[link]["Future_estimation_error"].tolist()

    return result

In [7]:
days_to_remove = 5
days_to_estimate = 5
pattern = r'_(\d{1,3}-\d{1,3})\.xlsx'
# pattern = r'sherbrooke_(\d{1,3}-\d{1,3})_max_\d+\.\d+\.xlsx'
normalize_data = False
save_figures=True
show_figs = False

result_dict = {}
files_in_dir = os.listdir('LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024')
save_path = f'LauraHandy/Results/Brisbane/RF_estimation/{days_to_estimate}_days/up_to_15-07-2024/'

data_files = []
for file_name in files_in_dir:
    data_files.append('LauraHandy/data/not_normalized/Brisbane/up_to_15-07-2024/' + file_name)
max_error = 0
for file_path in data_files:
    data = pd.read_excel(file_path)
    link = re.search(pattern, file_path).group(1)
    if max(data['Error']) > max_error:
        max_error = max(data['Error'])
    if normalize_data:
        save_path_fig = f'{save_path}Normalized/Link_{link}.png'
    else:
        save_path_fig = f'{save_path}/Not_normalized/Link_{link}.png'

    result_dict.update(get_error_estimation_rf(data, days_to_remove, days_to_estimate, link, normalize_data, save_figures, save_path_fig, show_figs))
    
if normalize_data:
    save_path_json = f'{save_path}Normalized/results.json'
else:
    save_path_json = f'{save_path}Not_normalized/results.json'

with open(save_path_json, "w") as outfile: 
    json.dump(result_dict, outfile)



Estimating for link 0-1
Estimating for link 0-14
Estimating for link 1-2
Estimating for link 10-11
Estimating for link 10-9
Estimating for link 100-101
Estimating for link 100-110
Estimating for link 100-99
Estimating for link 101-102
Estimating for link 102-103
Estimating for link 102-92
Estimating for link 103-104
Estimating for link 104-105
Estimating for link 104-111
Estimating for link 105-106
Estimating for link 106-107
Estimating for link 106-93
Estimating for link 107-108
Estimating for link 108-112
Estimating for link 109-114
Estimating for link 109-96
Estimating for link 11-12
Estimating for link 110-118
Estimating for link 111-122
Estimating for link 112-126
Estimating for link 113-114
Estimating for link 114-115
Estimating for link 115-116
Estimating for link 116-117
Estimating for link 117-118
Estimating for link 118-119
Estimating for link 119-120
Estimating for link 12-13
Estimating for link 12-17
Estimating for link 120-121
Estimating for link 121-122
Estimating for lin